In [1]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd  # A principal biblioteca de Data Science para tabelas (DataFrames) e manipulação matricial.

# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [2]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE FONTES DE DADOS E INFRAESTRUTURA DE NUVEM
# ==============================================================================

# --- FONTE 1: REPOSITÓRIO REMOTO (GITHUB) ---
# URL base onde estão armazenados arquivos CSV públicos contendo dados históricos de filmes.
url_github_movies_dataset = "https://media.githubusercontent.com/media/lucas-aulas/dataset-movies/refs/heads/main"
# --- FONTE 2: AMBIENTE DE LEITURA (PROJETO COMPARTILHADO DO PROFESSOR) ---
# ID do projeto na Google Cloud de onde leremos uma tabela institucional de bilheterias.
fonte_gcp_project_id = "meu-projeto-eng-dados"

# --- DESTINO: AMBIENTE DE ESCRITA (PROJETO PRIVADO DO ALUNO) ---
# Caminho local do arquivo JSON que contém as credenciais de segurança (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público! Ela equivale à senha do seu banco de dados.
meu_service_account_file = "C:\\Users\\1558989\\Downloads\\projeto-ceci-eng-dados-25dd45026529.json"
meu_gcp_project_id = "projeto-ceci-eng-dados"  # ID do seu projeto pessoal criado no GCP.

# Inicialização da autenticação e do cliente BigQuery
# Transforma o arquivo JSON em um objeto de credenciais aceito pelo Google.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# O objeto 'client' funciona como a nossa ponte ativa de conexão/comunicação com a nuvem do Google.
bigquery_client = bigquery.Client(credentials=gcp_credentials)

In [3]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# Leitura de um arquivo CSV hospedado na web diretamente para a memória como um DataFrame do Pandas.
# Este dataset contém dados gerais de filmes avaliados no site IMDB (Internet Movie Database).
df_imdb_filmes = pd.read_csv(f"{url_github_movies_dataset}/imdb_filmes.csv")

In [4]:
df_imdb_filmes.head(4)


,Unnamed: 0,ID,TITLE,LINK,YEAR,DURATION,RATING_MPA,RATING_IMDB,VOTE,BUDGET,GROSS_WORLD_WIDE,GROSS_US_CANADA,GROSS_OPENING_WEEKEND,DIRECTOR,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
0,0,tt0073470,Strip Nude for Your Killer,https://www.imdb.com/title/tt0073470,1975,1h 38m,Not Rated,5.6,33000.0,NaN,NaN,NaN,NaN,Andrea Bianchi,"Andrea Bianchi, Massimo Felisatti","Edwige Fenech, Nino Castelnuovo, Femi Benussi","Erotic Thriller, Giallo, Slasher Horror, Horro...",Italy,"Milan, Lombardia, Italy",FRAL Cinematografica,Italian,0,0,0
1,1,tt0072764,Carry on Behind,https://www.imdb.com/title/tt0072764,1975,1h 30m,Not Rated,5.6,31000.0,NaN,NaN,NaN,NaN,Gerald Thomas,Dave Freeman,"Elke Sommer, Kenneth Williams, Bernard Bresslaw","Parody, Slapstick, Comedy",United Kingdom,"Bad Godesberg Way, Maidenhead, Berkshire, Engl...","The Rank Organisation, Peter Rogers Productions",English,0,0,0
2,2,tt0131526,"So Young, So Lovely, So Vicious...",https://www.imdb.com/title/tt0131526,1975,1h 30m,NaN,4.9,487.0,NaN,NaN,NaN,NaN,Silvio Amadio,"Silvio Amadio, Roberto Natale","Gloria Guida, Dagmar Lassander, Fred Robsahm","Drama, Mystery, Thriller",Italy,"Sardinia, Italy",Domiziana Internazionale Cinematografica,Italian,0,0,0
3,3,tt0073335,Mahogany,https://www.imdb.com/title/tt0073335,1975,1h 49m,PG,6.1,23000.0,NaN,NaN,5000000.0,NaN,"Berry Gordy, Tony Richardson, Jack Wormser","John Byrum, Toni Amber","Diana Ross, Billy Dee Williams, Anthony Perkins","Drama, Romance",United States,Marshall Field Co Department Store 111 N Sta...,"Motown Productions, Nikor Productions, Paramou...","English, Italian",0,0,1


In [5]:
# Execução de uma consulta SQL estruturada diretamente dentro do Data Warehouse (BigQuery) da nuvem.
# O método '.to_dataframe()' faz o download do resultado da query SQL e o converte em uma tabela Pandas local.
# 'create_bqstorage_client=False' evita criar uma conexão otimizada adicional, mantendo a chamada simples.
df_boxoffice_dc_marvel = bigquery_client.query(
    f"SELECT * FROM {fonte_gcp_project_id}.raw.boxoffice_dc_marvel"
).to_dataframe(create_bqstorage_client=False)

In [6]:
df_boxoffice_dc_marvel.head(4)


,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BOX_OFFICE_GROSS_WORLDWIDE,BUDGET,MCU,PHASE,DISTRIBUTOR,MPAA_RATING,LENGTH,MINUTES,FRANCHISE,CHARACTER_FAMILY,DOMESTIC,GROSS_TO_BUDGET,ROTTEN_TOMATOES_CRITIC_SCORE,MALE_FEMALE_LED,YEAR,INFLATION_ADJUSTED_WORLDWIDE_GROSS,INFLATION_ADJUSTED_BUDGET,25X_PROD,BREAK_EVEN
0,113,NaN,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN
1,89,The New Mutants,2020-08-28,23852659,25316935,49169594,67000000,False,NaN,20th Century Studios,PG-13,01:34:00,94.0,Marvel,X-Men,0.49,0.73,36.0,Co-starring,2020.0,49169594,67000000,167500000,Flop
2,98,Morbius,2022-04-01,73865530,93595431,167460961,75000000,False,NaN,Columbia Pictures,PG-13,01:44:00,104.0,Marvel,Spider-Man Allies and Villains,0.44,2.23,15.0,Male,2022.0,167460961,75000000,187500000,Flop
3,112,Madame Web,2024-02-14,42619699,54000000,96619699,80000000,False,NaN,Columbia Pictures,PG-13,01:56:00,116.0,Marvel,Spider-Man Allies and Villains,0.44,1.21,13.0,Female,2024.0,96619699,80000000,200000000,Flop


In [7]:
# ==============================================================================
# 7. CARGA INICIAL - CAMADA BRONZE (LOAD)
# ==============================================================================

# Enviando os dados brutos locais para tabelas físicas dentro do BigQuery.
# 'destination_table="bronze.nome"' cria ou insere os dados no Dataset chamado 'bronze'.
# 'if_exists="replace"' indica que se a tabela já existir lá em cima, ela será apagada e recriada do zero.
import pandas_gbq


pandas_gbq.to_gbq(
    df_imdb_filmes,
    project_id=meu_gcp_project_id,
    destination_table="bronze.imdb_filmes",
    if_exists="replace",
    credentials=gcp_credentials,
)

pandas_gbq.to_gbq(
    df_boxoffice_dc_marvel,
    project_id=meu_gcp_project_id,
    destination_table="bronze.boxoffice_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)